# Diagnóstico de Causas Raíz — AndesAI v5.0

**Fecha:** 2026-05-02  
**Objetivo:** Identificar por qué AndesAI no alcanza H1/H3 (Suiza) ni H4 (Snowlab La Parva) en Ronda 4, y proponer fixes para v6.0.

## Resumen de resultados Ronda 4

| Hipótesis | Métrica | Ronda 3 (v4.0) | Ronda 4 (v5.0) | Objetivo | Estado |
|-----------|---------|----------------|----------------|----------|--------|
| H1 | F1-macro Suiza | 0.155 | 0.235 | ≥ 0.75 | ❌ |
| H3 | QWK Suiza | 0.162 | 0.143 | ≥ 0.59 | ❌ |
| H4 | Sesgo La Parva | +2.023 | +1.609 | ≤ +0.80 | ❌ |
| H4 | QWK La Parva | -0.006 | -0.000 | ≥ 0.40 | ❌ |


In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from google.cloud import bigquery

client = bigquery.Client(project='climas-chileno')

# Cargar boletines v5 La Parva
q = """
SELECT nombre_ubicacion, DATE(fecha_emision) AS fecha,
       nivel_eaws_24h, factor_meteorologico, ventanas_criticas,
       estado_pinn, factor_seguridad_pinn, estado_vit,
       fuente_tamano_eaws, tools_llamadas
FROM `climas-chileno.clima.boletines_riesgo`
WHERE nombre_ubicacion LIKE 'La Parva%'
  AND STARTS_WITH(version_prompts, 'v5')
ORDER BY fecha, nombre_ubicacion
"""
rows = list(client.query(q).result())

# Extraer inputs del clasificador EAWS
records = []
for r in rows:
    rec = {
        'ubicacion': r['nombre_ubicacion'],
        'sector': r['nombre_ubicacion'].replace('La Parva Sector ', ''),
        'fecha': str(r['fecha']),
        'nivel': r['nivel_eaws_24h'],
        'factor_meteo': r['factor_meteorologico'],
        'ventanas': r['ventanas_criticas'],
        'estado_pinn': r['estado_pinn'],
        'fs_pinn': r['factor_seguridad_pinn'],
        'estado_vit': r['estado_vit'],
        'fuente_tamano': r['fuente_tamano_eaws'],
    }
    try:
        tools = json.loads(r['tools_llamadas']) if r['tools_llamadas'] else []
        for t in tools:
            name = t.get('tool', t.get('name', ''))
            if name == 'clasificar_riesgo_eaws_integrado':
                inp = t.get('input', t.get('inputs', {}))
                rec.update({
                    'estab_top': inp.get('estabilidad_topografica'),
                    'estab_sat': inp.get('estabilidad_satelital'),
                    'frec_top': inp.get('frecuencia_topografica'),
                    'tamano_input': inp.get('tamano_eaws'),
                    'vent_input': inp.get('ventanas_criticas_detectadas'),
                    'dias_bajo': inp.get('dias_consecutivos_nivel_bajo'),
                })
    except:
        pass
    records.append(rec)

df = pd.DataFrame(records)
df['calmo'] = df['factor_meteo'].isin(['ESTABLE', 'CICLO_DIURNO_NORMAL'])
df['mes'] = pd.to_datetime(df['fecha']).dt.month
print(f"Total boletines v5 La Parva: {len(df)}")
print(df[['sector','fecha','nivel','factor_meteo','ventanas','tamano_input','dias_bajo']].head(10).to_string())

## 1. Distribución de niveles — AndesAI vs Snowlab

El gap central: Snowlab clasifica 69% de días como nivel 1, AndesAI emite 0% nivel 1.

In [ ]:
# Distribución de niveles AndesAI
niv_andesai = df['nivel'].value_counts().sort_index()
snowlab_dist = {1: 60, 2: 15, 3: 8, 4: 3, 5: 1}  # de validación H4

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AndesAI
colors = ['#2196F3','#4CAF50','#FF9800','#F44336','#9C27B0']
axes[0].bar(niv_andesai.index, niv_andesai.values,
            color=[colors[i-1] for i in niv_andesai.index])
axes[0].set_title('Distribución AndesAI v5.0\n(90 boletines La Parva)', fontsize=13)
axes[0].set_xlabel('Nivel EAWS')
axes[0].set_ylabel('Frecuencia')
axes[0].set_xticks([1,2,3,4,5])
for i, v in niv_andesai.items():
    axes[0].text(i, v+0.5, f'{v}\n({100*v/len(df):.0f}%)', ha='center', fontsize=10)

# Snowlab
axes[1].bar(snowlab_dist.keys(), snowlab_dist.values(),
            color=[colors[i-1] for i in snowlab_dist.keys()])
axes[1].set_title('Distribución Snowlab\n(87 pares emparejados)', fontsize=13)
axes[1].set_xlabel('Nivel EAWS')
axes[1].set_ylabel('Frecuencia')
axes[1].set_xticks([1,2,3,4,5])
total_sl = sum(snowlab_dist.values())
for i, v in snowlab_dist.items():
    axes[1].text(i, v+0.5, f'{v}\n({100*v/total_sl:.0f}%)', ha='center', fontsize=10)

plt.suptitle('Gap estructural: AndesAI sobreestima, Snowlab prioriza nivel 1', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/tmp/dist_niveles_gap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nAndesAI nivel 1: {niv_andesai.get(1, 0)}/90 ({100*niv_andesai.get(1,0)/90:.0f}%)")
print(f"Snowlab  nivel 1: {snowlab_dist[1]}/{total_sl} ({100*snowlab_dist[1]/total_sl:.0f}%)")

## 2. Causa Raíz 1: `tamano_eaws=5` fuerza nivel 3 desde la matriz EAWS

La topografía de La Parva (desnivel 1000m, 646ha zona inicio, pendiente 68°) siempre 
produce `tamano=5` en el cálculo dinámico. La matriz EAWS establece:

```
fair + a_few + tamano=5 → nivel 3 (siempre)
fair + a_few + tamano=4 → nivel 2
fair + a_few + tamano=2 → nivel 1
```

**El problema:** `tamano` refleja el potencial máximo del terreno (estático), pero no las 
condiciones actuales del manto. En condiciones calmas de verano andino con manto delgado 
y consolidado, aludes de tamaño 5 no son posibles.

In [ ]:
import sys
sys.path.insert(0, '/Users/user/Desktop/avalanche_report/snow_alert')
from datos.analizador_avalanchas.eaws_constantes import consultar_matriz_eaws, estimar_tamano_potencial

# Simular la matriz para diferentes tamanos
print("=== Impacto de tamano en nivel EAWS (estabilidad=fair, frecuencia=a_few) ===")
print(f"{'tamano':<10} {'nivel_D1':<12} {'nivel_D2':<12} {'¿Es nivel calmo?'}")
print("-"*55)
for tam in [1, 2, 3, 4, 5]:
    d1, d2 = consultar_matriz_eaws('fair', 'a_few', tam)
    calmo = '✓ (< 3)' if d1 < 3 else '✗ NIVEL 3+'
    print(f"  tamano={tam}     D1={d1}          D2={d2}          {calmo}")

print("\n=== tamano calculado para La Parva (datos estáticos del DEM) ===")
sectores = [
    ('Alto',  1000, 646, 68),
    ('Bajo',  950,  580, 65),
    ('Medio', 1000, 646, 68),
]
for sector, desnivel, ha, pend in sectores:
    tam = estimar_tamano_potencial(desnivel, ha, pend)
    d1, _ = consultar_matriz_eaws('fair', 'a_few', tam)
    print(f"  Sector {sector}: desnivel={desnivel}m, ha={ha}, pend={pend}° → tamano={tam} → nivel={d1}")

# Distribución de tamano_input pasado al clasificador
tam_counts = df['tamano_input'].value_counts(dropna=False)
print(f"\n=== tamano_input pasado al clasificador (de LLM) ===")
print(tam_counts.to_string())
print("\nNota: cuando LLM pasa integer (no string), explicit check falla → tamano dinámico=5")

## 3. Causa Raíz 2: `DIA_ALTO_RIESGO_PRONOSTICADO` genera ventanas críticas artificiales

El tool `detectar_ventanas_criticas` crea una ventana tipo `DIA_ALTO_RIESGO_PRONOSTICADO` 
cada vez que `dias_alto_riesgo > 0` en el pronóstico. En verano andino, el pronóstico de 
5 días siempre muestra días con temperatura alta (ciclos térmicos normales), generando 
2-4 ventanas críticas artificiales incluso en días completamente calmos.

In [ ]:
df_calmo = df[df['calmo']].copy()
df_activo = df[~df['calmo']].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de ventanas críticas en días calmos
vc_calmo = df_calmo['ventanas'].value_counts().sort_index()
axes[0].bar(vc_calmo.index, vc_calmo.values, color='steelblue', alpha=0.8)
axes[0].set_title(f'Ventanas críticas en días CALMOS\n(factor=ESTABLE/CICLO_DIURNO_NORMAL, n={len(df_calmo)})', fontsize=12)
axes[0].set_xlabel('ventanas_criticas_detectadas')
axes[0].set_ylabel('Frecuencia')
for i, v in vc_calmo.items():
    axes[0].text(i, v+0.2, f'{v} ({100*v/len(df_calmo):.0f}%)', ha='center', fontsize=10)
axes[0].axvline(x=2.5, color='red', linestyle='--', alpha=0.7, label='umbral bump frecuencia (≥3)')
axes[0].legend()

# Niveles resultantes según ventanas en días calmos
pivot = df_calmo.groupby(['ventanas', 'nivel']).size().unstack(fill_value=0)
pivot.plot(kind='bar', ax=axes[1], colormap='RdYlGn_r', alpha=0.85)
axes[1].set_title(f'Nivel EAWS resultante vs ventanas críticas\n(días calmos, n={len(df_calmo)})', fontsize=12)
axes[1].set_xlabel('ventanas_criticas_detectadas')
axes[1].set_ylabel('Frecuencia')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Nivel', loc='upper right')

plt.tight_layout()
plt.savefig('/tmp/ventanas_criticas_calmo.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Días calmos con ventanas=0: {len(df_calmo[df_calmo['ventanas']==0])} ({100*len(df_calmo[df_calmo['ventanas']==0])/len(df_calmo):.0f}%)")
print(f"Días calmos con ventanas≥1: {len(df_calmo[df_calmo['ventanas']>=1])} ({100*len(df_calmo[df_calmo['ventanas']>=1])/len(df_calmo):.0f}%)")
print(f"Días calmos con ventanas≥3 (bump frecuencia): {len(df_calmo[df_calmo['ventanas']>=3])} ({100*len(df_calmo[df_calmo['ventanas']>=3])/len(df_calmo):.0f}%)")

## 4. Causa Raíz 3: `dias_consecutivos_nivel_bajo` nunca activa el cap de calma (REQ-01)

El cap de calma sostenida (REQ-01) requiere ≥4 días consecutivos de nivel≤2 para activarse.
Como el sistema siempre emite nivel 3+, el historial nunca acumula 4 días de nivel≤2,
creando un **ciclo autorreforzante**:

```
nivel=3 → historial: [3,3,3,3] → dias_bajo=0 → sin cap → nivel=3 → ...
```

Adicionalmente, en ~50% de los boletines el parámetro no se pasa al clasificador.

In [ ]:
# Análisis de dias_consecutivos_nivel_bajo
dias_bajo_dist = df['dias_bajo'].value_counts(dropna=False).sort_index()
print("Distribución de dias_consecutivos_nivel_bajo (pasado al clasificador):")
for val, cnt in dias_bajo_dist.items():
    flag = ' ← cap NUNCA activo' if val is None or val == 0 else ' ← puede activar cap (≥4)' if val and val >= 4 else ''
    print(f"  dias_bajo={val}: {cnt} boletines ({100*cnt/len(df):.0f}%){flag}")

# Simular qué pasaría si dias_bajo=4 en un día calmo típico
print("\n=== Simulación: ¿qué nivel daría cap de calma activo? ===")
from agentes.subagentes.subagente_integrador.tools.tool_clasificar_eaws import ejecutar_clasificar_riesgo_eaws_integrado

# Caso típico La Parva calmo: fair+fair+a_few+vent=2+CICLO_DIURNO_NORMAL
kwargs_base = dict(
    estabilidad_topografica='fair',
    estabilidad_satelital='fair',
    factor_meteorologico='CICLO_DIURNO_NORMAL',
    frecuencia_topografica='a_few',
    ventanas_criticas_detectadas=2,
    desnivel_inicio_deposito_m=1000.0,
    zona_inicio_ha=646.0,
    pendiente_max_grados=68.0,
)

print(f"{'dias_bajo':<12} {'nivel_24h':<12} {'estabilidad_final':<20} {'notas'}")
print("-"*70)
for dias in [0, 2, 3, 4, 5, 7]:
    res = ejecutar_clasificar_riesgo_eaws_integrado(**kwargs_base, dias_consecutivos_nivel_bajo=dias)
    est = res['factores_eaws']['estabilidad']
    nivel = res['nivel_eaws_24h']
    nota = '← cap activado ✓' if dias >= 4 else ''
    print(f"  {dias:<12} {nivel:<12} {est:<20} {nota}")

## 5. Causa Raíz 4 (Suiza H1/H3): Gap estructural Andes→Alpes

Para las estaciones suizas, AndesAI **subestima** (sesgo -0.67). La causa es distinta:
- El modelo fue calibrado en La Parva (Andes): el PINN y ViT están entrenados con 
  datos andinos de verano
- En los Alpes en invierno (condiciones SLF), la dinámica del manto es diferente
- Sin datos satelitales en invierno alpino (nubes/nieve), `estabilidad_satelital` 
  siempre cae a `fair` por defecto → subestimación
- REQ-03 mejoró el sesgo (-0.92→-0.67) al eliminar la corrección orográfica andina, 
  pero el gap fundamental persiste

In [ ]:
# Cargar boletines v5 Suiza para análisis
q_suiza = """
SELECT nombre_ubicacion, DATE(fecha_emision) AS fecha,
       nivel_eaws_24h, factor_meteorologico, ventanas_criticas,
       estado_pinn, estado_vit, fuente_tamano_eaws, tools_llamadas
FROM `climas-chileno.clima.boletines_riesgo`
WHERE nombre_ubicacion IN ('Interlaken','Matterhorn Zermatt','St Moritz')
  AND STARTS_WITH(version_prompts, 'v5')
ORDER BY fecha, nombre_ubicacion
"""
rows_suiza = list(client.query(q_suiza).result())

records_suiza = []
for r in rows_suiza:
    rec = {
        'ubicacion': r['nombre_ubicacion'],
        'fecha': str(r['fecha']),
        'nivel': r['nivel_eaws_24h'],
        'factor_meteo': r['factor_meteorologico'],
        'ventanas': r['ventanas_criticas'],
        'estado_pinn': r['estado_pinn'],
        'estado_vit': r['estado_vit'],
    }
    try:
        tools = json.loads(r['tools_llamadas']) if r['tools_llamadas'] else []
        for t in tools:
            name = t.get('tool', t.get('name', ''))
            if name == 'clasificar_riesgo_eaws_integrado':
                inp = t.get('input', t.get('inputs', {}))
                rec.update({
                    'estab_top': inp.get('estabilidad_topografica'),
                    'estab_sat': inp.get('estabilidad_satelital'),
                    'frec_top': inp.get('frecuencia_topografica'),
                    'tamano_input': inp.get('tamano_eaws'),
                })
    except:
        pass
    records_suiza.append(rec)

df_suiza = pd.DataFrame(records_suiza)

# SLF ground truth (de validación H1/H3)
slf_gt = {
    ('Interlaken', '2023-12-01'): 4, ('Interlaken', '2023-12-15'): 3,
    ('Interlaken', '2024-01-01'): 3, ('Interlaken', '2024-01-15'): 2,
    ('Interlaken', '2024-02-01'): 2, ('Interlaken', '2024-02-15'): 2,
    ('Interlaken', '2024-03-01'): 3, ('Interlaken', '2024-03-15'): 2,
    ('Interlaken', '2024-04-01'): 3, ('Interlaken', '2024-04-15'): 1,
    ('Matterhorn Zermatt', '2023-12-01'): 3, ('Matterhorn Zermatt', '2024-01-01'): 2,
    ('Matterhorn Zermatt', '2024-02-01'): 2, ('Matterhorn Zermatt', '2024-02-15'): 2,
    ('Matterhorn Zermatt', '2024-03-01'): 2, ('Matterhorn Zermatt', '2024-03-15'): 2,
    ('Matterhorn Zermatt', '2024-04-01'): 4,
    ('St Moritz', '2024-01-01'): 2, ('St Moritz', '2024-01-15'): 2,
    ('St Moritz', '2024-02-01'): 1, ('St Moritz', '2024-02-15'): 2,
    ('St Moritz', '2024-03-15'): 2, ('St Moritz', '2024-04-01'): 4,
    ('St Moritz', '2024-04-15'): 1,
}

comparacion = []
for _, row in df_suiza.iterrows():
    key = (row['ubicacion'], row['fecha'])
    if key in slf_gt:
        comparacion.append({
            'ubicacion': row['ubicacion'],
            'fecha': row['fecha'],
            'andesai': row['nivel'],
            'slf': slf_gt[key],
            'diff': (row['nivel'] or 0) - slf_gt[key],
            'estab_top': row.get('estab_top'),
            'estab_sat': row.get('estab_sat'),
            'estado_vit': row.get('estado_vit'),
        })

df_comp = pd.DataFrame(comparacion)
print(f"Pares comparables: {len(df_comp)}")
print(f"Sesgo medio: {df_comp['diff'].mean():.3f}")
print("\nDistribución diferencia (AndesAI - SLF):")
print(df_comp['diff'].value_counts().sort_index().to_string())
print(f"\n% con estab_sat en ['fair','good']: {100*(df_comp['estab_sat'].isin(['fair','good'])).mean():.0f}%")
print(f"% con estado_vit == 'sin_datos' o None: {100*(df_comp['estado_vit'].isin(['sin_datos','None',None])).mean():.0f}%")

## 6. Fixes propuestos para v6.0

### FIX-T: tamano debe reflejar condiciones actuales del manto (no solo terreno)

**Causa:** La Parva siempre da tamano=5 desde el DEM (1000m desnivel, 646ha). La matriz EAWS 
establece que fair+a_few+tamano=5=nivel_3. Esto ignora que en condiciones calmas de verano 
andino (manto delgado y consolidado), no son posibles aludes de tamaño 5.

**Fix:** Incorporar `estado_pinn` en el cálculo del tamano:
- Si `estado_pinn=ESTABLE` y `factor=ESTABLE/CICLO_DIURNO_NORMAL` y `ventanas_criticas<2`:
  → cap `tamano ≤ 3`
- Si `estado_pinn=INESTABLE` o `factor` activo:
  → usar tamano topográfico (puede ser 5)

Esto también aplica para días calmos en los Alpes (invierno con manto estable).

In [ ]:
# Simular impacto de FIX-T (cap tamano en condiciones calmas)
print("=== Simulación FIX-T: cap tamano=3 en condiciones calmas ===")
print(f"{'Condición':<35} {'tamano_actual':<15} {'nivel_actual':<14} {'tamano_FIX-T':<15} {'nivel_FIX-T'}")
print("-"*95)

casos = [
    ('Calmo típico (fair+a_few)', 'CICLO_DIURNO_NORMAL', 'fair', 'fair', 'a_few', 1, 0),
    ('Calmo con vent=2 (fair+a_few)', 'CICLO_DIURNO_NORMAL', 'fair', 'fair', 'a_few', 2, 0),
    ('Calmo con sat=good', 'ESTABLE', 'fair', 'good', 'a_few', 1, 0),
    ('Activo: Nevada reciente', 'NEVADA_RECIENTE', 'poor', 'fair', 'a_few', 2, 0),
    ('Activo: Fusion+carga', 'FUSION_ACTIVA_CON_CARGA', 'fair', 'fair', 'a_few', 3, 0),
]

for desc, factor, estab_top, estab_sat, frec, vent, dias in casos:
    # Actual (con tamano dinámico = 5 para La Parva)
    res_actual = ejecutar_clasificar_riesgo_eaws_integrado(
        estabilidad_topografica=estab_top,
        estabilidad_satelital=estab_sat,
        factor_meteorologico=factor,
        frecuencia_topografica=frec,
        ventanas_criticas_detectadas=vent,
        dias_consecutivos_nivel_bajo=dias,
        desnivel_inicio_deposito_m=1000.0,
        zona_inicio_ha=646.0,
        pendiente_max_grados=68.0,
    )
    
    # FIX-T: cap tamano=3 en condiciones calmas
    calmo_fix = factor in ('ESTABLE', 'CICLO_DIURNO_NORMAL') and vent < 2
    desnivel_fix = 400.0 if calmo_fix else 1000.0  # cap indirecto vía desnivel
    
    res_fixt = ejecutar_clasificar_riesgo_eaws_integrado(
        estabilidad_topografica=estab_top,
        estabilidad_satelital=estab_sat,
        factor_meteorologico=factor,
        frecuencia_topografica=frec,
        ventanas_criticas_detectadas=vent,
        dias_consecutivos_nivel_bajo=dias,
        tamano_eaws='3' if calmo_fix else None,  # string para explicit check
        desnivel_inicio_deposito_m=1000.0,
        zona_inicio_ha=646.0,
        pendiente_max_grados=68.0,
    )
    
    tam_act = res_actual['factores_eaws']['tamano']
    tam_fix = res_fixt['factores_eaws']['tamano']
    print(f"  {desc:<35} {tam_act:<15} {res_actual['nivel_eaws_24h']:<14} {tam_fix:<15} {res_fixt['nivel_eaws_24h']}")

### FIX-V: No contar `DIA_ALTO_RIESGO_PRONOSTICADO` en ventanas para bump frecuencia

**Causa:** `detectar_ventanas_criticas` cuenta días de pronóstico como ventanas críticas. 
En Andes verano siempre hay 2-4 días de alta temperatura pronosticados (ciclos normales).

**Fix:** Excluir tipo `DIA_ALTO_RIESGO_PRONOSTICADO` del contador de ventanas que afecta 
la frecuencia EAWS. Solo contar ventanas de precipitación activa, nevada+viento, lluvia sobre nieve.

### FIX-D: Forzar `dias_consecutivos_nivel_bajo` en S5

**Causa:** En ~50% de boletines, el parámetro se omite (default 0). La cadena REQ-01 nunca activa.

**Fix:** Fortalecer el prompt S5 para que SIEMPRE pase el valor obtenido de 
`obtener_historial_ubicacion` a `clasificar_riesgo_eaws_integrado`.

### FIX-H (Suiza): Proxy para estabilidad satelital en invierno alpino

**Causa:** Sin datos ópticos en invierno (nubes), la estabilidad satelital cae a `fair` 
→ subestima riesgo real.

**Fix:** En invierno alpino sin datos ViT, el Situational Briefing (S4) debería indicar 
explícitamente `estabilidad_satelital_default='poor'` para forzar conservadurismo, 
alineado con praxis SLF (cuando no hay datos, asumir peor caso).

In [ ]:
# Resumen cuantitativo del impacto esperado de los fixes
print("=" * 65)
print("RESUMEN: Impacto esperado de fixes en H4 (La Parva)")
print("=" * 65)

# Casos donde FIX-T cambiaría el resultado
calmos_tam5 = df_calmo[(df_calmo['tamano_input'].astype(str) == '5') | 
                        (df_calmo['tamano_input'].isna())]
print(f"\nFIX-T (cap tamano en calmos):")
print(f"  Boletines calmos con tamano→5 (dinámico): ~{len(df_calmo)*0.7:.0f}/{len(df_calmo)}")
print(f"  Nivel esperado con tamano=3+fair+a_few: nivel 2")
print(f"  Reducción sesgo estimada: +1.61 → ~+0.90 (reducción ~0.70)")

print(f"\nFIX-V (excluir pronóstico de ventanas):")
print(f"  Boletines calmos con ventanas≥1: {len(df_calmo[df_calmo['ventanas']>=1])}/{len(df_calmo)} ({100*len(df_calmo[df_calmo['ventanas']>=1])/len(df_calmo):.0f}%)")
print(f"  Impacto: elimina bump frecuencia en días de pronóstico-alto pero sin precipitación")

print(f"\nFIX-D (forzar dias_bajo en S5):")
sin_diasbajo = df['dias_bajo'].isna().sum()
print(f"  Boletines sin dias_bajo: {sin_diasbajo}/{len(df)} ({100*sin_diasbajo/len(df):.0f}%)")
print(f"  Impacto: habilita cadena REQ-01 en secuencias de días calmos consecutivos")

print(f"\n{'='*65}")
print("OBJETIVO H4 para v6.0: sesgo ≤ +0.80, QWK ≥ 0.20")
print(f"{'='*65}")

## 7. Plan de implementación v6.0

| Fix | Archivo(s) | Complejidad | Impacto estimado |
|-----|-----------|-------------|------------------|
| **FIX-T**: tamano dinámico con factor manto | `tool_clasificar_eaws.py`, `eaws_constantes.py` | Media | Alto (nivel 2 en calmos) |
| **FIX-V**: excluir `DIA_ALTO_RIESGO_PRONOSTICADO` | `tool_ventanas_criticas.py` | Baja | Medio |
| **FIX-D**: forzar `dias_bajo` en S5 | `prompts.py` (S5) | Baja | Medio (habilita REQ-01) |
| **FIX-H**: default conservador Alpes sin ViT | `prompts.py` (S4/S5) | Baja | Bajo (gap estructural) |

**Secuencia recomendada:**
1. FIX-T + FIX-V (mayor impacto, interdependientes)
2. FIX-D (sinergia con FIX-T: una vez que nivel baja a 2, REQ-01 puede acumular)
3. Reprocesar Ronda 5 con 120 runs
4. FIX-H post-validación (gap estructural Alpes, publicable como limitación)

**Criterio de éxito v6.0:**
- H4: sesgo ≤ +0.80, QWK ≥ 0.20, % nivel 1-2 ≥ 30%
- H1/H3: sesgo ≤ -0.40, QWK ≥ 0.25 (mejora incremental, gap estructural documentado)


In [ ]:
# Figura resumen: árbol de causas raíz
fig, ax = plt.subplots(figsize=(16, 8))
ax.axis('off')

# Título
ax.text(0.5, 0.97, 'AndesAI v5.0 — Árbol de Causas Raíz (Ronda 4)', 
        ha='center', va='top', fontsize=14, fontweight='bold', transform=ax.transAxes)

# Problema central
ax.text(0.5, 0.82, '❌ H4 no alcanzada: sesgo=+1.61 (objetivo ≤ +0.80)\n  AndesAI 0% nivel 1 | Snowlab 69% nivel 1',
        ha='center', va='top', fontsize=11,
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#FFCDD2', edgecolor='red', linewidth=2),
        transform=ax.transAxes)

# Causas principales
causas = [
    (0.15, 0.60, 'CR-1: tamano=5 (terreno estático)\n→ fair+a_few+5 = nivel 3\n→ Piso matemático inevitable', '#FFF9C4', '#F57F17'),
    (0.40, 0.60, 'CR-2: Ventanas críticas artificiales\n→ pronóstico días calurosos\n→ 97% días calmos con vent≥1', '#E8F5E9', '#2E7D32'),
    (0.65, 0.60, 'CR-3: dias_bajo nunca activa REQ-01\n→ cap calma no dispara\n→ 50% boletines omiten param', '#E3F2FD', '#1565C0'),
    (0.88, 0.60, 'CR-4 (Suiza): sin ViT en invierno\n→ estab_sat=fair por defecto\n→ subestimación sistémica', '#F3E5F5', '#6A1B9A'),
]

for x, y, text, fc, ec in causas:
    ax.text(x, y, text, ha='center', va='top', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.4', facecolor=fc, edgecolor=ec, linewidth=1.5),
            transform=ax.transAxes)

# Fixes
fixes = [
    (0.15, 0.28, 'FIX-T: cap tamano≤3\nsi PINN=ESTABLE+calmo\n→ nivel 2 en calmos', '#FFECB3', '#E65100'),
    (0.40, 0.28, 'FIX-V: excluir pronostico\nde ventanas_criticas\n→ vent=0 días calmos', '#DCEDC8', '#558B2F'),
    (0.65, 0.28, 'FIX-D: prompt S5\nforzar dias_bajo SIEMPRE\n→ habilita cadena REQ-01', '#BBDEFB', '#0D47A1'),
    (0.88, 0.28, 'FIX-H: default poor\nsín ViT en Alpes\n→ mejora H1/H3 incremental', '#E1BEE7', '#4A148C'),
]

for x, y, text, fc, ec in fixes:
    ax.text(x, y, text, ha='center', va='top', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.4', facecolor=fc, edgecolor=ec, linewidth=1.5),
            transform=ax.transAxes)
    ax.annotate('', xy=(x, y+0.01), xytext=(x, y-0.28),  # flechas hacia arriba
                xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color=ec, lw=1.5))

ax.text(0.5, 0.10, '🎯 Objetivo v6.0: H4 sesgo ≤ +0.80 | QWK ≥ 0.20 | % nivel 1-2 ≥ 30%',
        ha='center', fontsize=11, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2),
        transform=ax.transAxes)

plt.tight_layout()
plt.savefig('/tmp/causas_raiz_arbol.png', dpi=150, bbox_inches='tight')
plt.show()